# Week 6 Spark Assignment  
## Spark Architecture and Efficient Data Processing

### Objective

Understand Spark architecture and perform efficient data processing using transformations, filtering, schema handling, and optimized file formats.

### Covered Topics

1. Spark Architecture
2. Driver, Cluster Manager, and Executors
3. Lazy Evaluation
4. DAG / Lineage Graph
5. CSV and Parquet file handling
6. Filtering and column selection
7. Renaming columns and casting data types
8. Adding calculated columns
9. Transformations and Actions
10. Predicate Pushdown
11. Safe output saving inside the `outputs` folder
12. Best practices for large datasets

## Notebook Execution Note

This notebook is designed for your current Windows folder structure:

`C:\Celebal Assignments\Celebal-Assignments\Week-6-Spark-Architecture-Data-Processing`

Your earlier error happened because Spark's native `.write.csv()` and `.write.parquet()` operations can fail on Windows when Hadoop `winutils.exe` is not configured correctly.

To make the notebook run smoothly, this file uses Spark for DataFrame processing and uses a safe Pandas-based output writer for saving final CSV and Parquet files into the `outputs` folder.

For this assignment-sized dataset, this approach is safe and avoids the Windows Hadoop write error.

## Setup Cell 1: Check and Install Required Python Packages

This cell checks whether required packages are available.  
If a package is missing, it installs it in the selected notebook kernel.

In [1]:
import sys
import subprocess
import importlib.util

required_packages = {
    "pyspark": "pyspark",
    "pandas": "pandas",
    "pyarrow": "pyarrow"
}

for import_name, package_name in required_packages.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"{package_name} is not installed. Installing now...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
    else:
        print(f"{package_name} is already installed.")

print("\nAll required Python packages are ready.")
print("Python executable:", sys.executable)

pyspark is already installed.
pandas is already installed.
pyarrow is already installed.

All required Python packages are ready.
Python executable: c:\Celebal Assignments\venv\Scripts\python.exe


## Setup Cell 2: Set Project Path

This cell moves the notebook execution to the correct project folder.

It also creates the required `data` and `outputs` folders if they do not already exist.

In [2]:
import os
import shutil
from pathlib import Path

# Your actual project folder
project_path = Path(r"C:\Celebal Assignments\Celebal-Assignments\Week-6-Spark-Architecture-Data-Processing")

# If this exact path exists, use it. Otherwise, use the current notebook folder.
if project_path.exists():
    os.chdir(project_path)
else:
    project_path = Path.cwd()
    os.chdir(project_path)

# Create required folders
Path("data").mkdir(exist_ok=True)
Path("outputs").mkdir(exist_ok=True)
Path("outputs/csv_outputs").mkdir(parents=True, exist_ok=True)
Path("outputs/parquet_outputs").mkdir(parents=True, exist_ok=True)
Path("outputs/final_outputs").mkdir(parents=True, exist_ok=True)

# Remove accidental placeholder folder if it exists from earlier testing
placeholder_folder = Path("outputs/csv_outputs/folder_name")
if placeholder_folder.exists():
    shutil.rmtree(placeholder_folder)

print("Current working directory:")
print(Path.cwd())

print("\nCurrent project items:")
print(os.listdir(Path.cwd()))

Current working directory:
C:\Celebal Assignments\Celebal-Assignments\Week-6-Spark-Architecture-Data-Processing

Current project items:
['data', 'insights.md', 'outputs', 'README.md', 'Week6_Spark_Assignment.ipynb', 'Week6_Spark_Assignment_Final.ipynb']


## Setup Cell 3: Create Sample CSV if Missing

The assignment requires reading data from:

`data/source.csv`

If the file already exists, this cell keeps it as it is.  
If it is missing, this cell creates a small sample dataset so the notebook can run without failing.

In [3]:
from pathlib import Path

source_file = Path("data/source.csv")

if not source_file.exists():
    sample_csv = '''product_id,product_name,category,price,old_name,status,amount,region,priority,base_price,user_id
P101,Laptop,Electronics,55000,Product_A,Completed,1200,North,Medium,50000,U001
P102,Phone,Electronics,25000,Product_B,Pending,800,South,High,22000,U002
P103,Table,Furniture,7000,Product_C,Completed,1500,East,Low,6500,
P104,Headphones,Electronics,3000,Product_D,Completed,2000,West,High,2800,U004
P105,Chair,Furniture,4500,Product_E,Cancelled,500,North,Low,4000,U005
'''
    source_file.write_text(sample_csv, encoding="utf-8")
    print("source.csv was missing, so a sample file was created.")
else:
    print("source.csv already exists.")

print("Source file path:", source_file.resolve())

source.csv already exists.
Source file path: C:\Celebal Assignments\Celebal-Assignments\Week-6-Spark-Architecture-Data-Processing\data\source.csv


## Setup Cell 4: Create Spark Session

A SparkSession is the entry point for working with Spark DataFrames.

This cell creates a local Spark session using all available CPU cores on your machine.

In [4]:
import os
import sys
import pandas as pd

# Use the same Python environment for PySpark worker and driver
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Stop old Spark session if the notebook was already run once
try:
    spark.stop()
except Exception:
    pass

spark = SparkSession.builder \
    .appName("Week 6 Spark Architecture and Data Processing") \
    .master("local[*]") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.sql.warehouse.dir", "file:///C:/temp/spark-warehouse") \
    .getOrCreate()

print("Spark Session created successfully.")
print("Spark version:", spark.version)
spark

Spark Session created successfully.
Spark version: 4.1.2


## Setup Cell 5: Safe Output Helper Functions

These helper functions save output directly into your `outputs` folder.

Why this is needed:

- Spark transformations will still be done using Spark DataFrames.
- Output saving is done through Pandas to avoid Windows Hadoop write errors.
- This keeps the notebook stable and prevents `Py4JJavaError` during file saving.

In [5]:
import os
import shutil
from pathlib import Path
import pandas as pd

def reset_folder(folder_path):
    folder = Path(folder_path)
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)
    return folder

def save_spark_df_as_single_csv(spark_df, folder_path, file_name):
    """
    Saves a Spark DataFrame as one clean CSV file.
    This avoids Spark native write errors on local Windows systems.
    """
    folder = reset_folder(folder_path)
    file_path = folder / file_name

    pandas_df = spark_df.toPandas()
    pandas_df.to_csv(file_path, index=False)

    print("CSV output saved successfully.")
    print("Folder:", folder_path)
    print("File:", file_name)
    print("Rows saved:", len(pandas_df))
    print("Files inside folder:", os.listdir(folder))

def save_spark_df_as_single_parquet(spark_df, folder_path, file_name):
    """
    Saves a Spark DataFrame as one Parquet file using Pandas + PyArrow.
    This avoids Spark native Parquet write errors on Windows.
    """
    folder = reset_folder(folder_path)
    file_path = folder / file_name

    pandas_df = spark_df.toPandas()
    pandas_df.to_parquet(file_path, index=False, engine="pyarrow")

    print("Parquet output saved successfully.")
    print("Folder:", folder_path)
    print("File:", file_name)
    print("Rows saved:", len(pandas_df))
    print("Files inside folder:", os.listdir(folder))

def read_parquet_as_spark_df(parquet_file_path):
    """
    Reads a Parquet file safely.
    First tries Spark read.
    If Spark read fails on Windows, it reads using Pandas and converts back to Spark DataFrame.
    """
    try:
        return spark.read.parquet(parquet_file_path)
    except Exception:
        print("Spark read.parquet failed on local Windows setup.")
        print("Using Pandas fallback and converting back to Spark DataFrame.")
        pandas_df = pd.read_parquet(parquet_file_path, engine="pyarrow")
        return spark.createDataFrame(pandas_df)

print("Safe output helper functions are ready.")

Safe output helper functions are ready.


## Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

In a Spark application, the **Driver** is the main program that controls the complete Spark job. It creates the SparkSession, builds the execution plan, and coordinates all the work.

The **Cluster Manager** provides resources such as CPU and memory to the Spark application. Examples include Spark Standalone, YARN, Mesos, and Kubernetes.

The **Executors** are worker processes that run the actual tasks. They process the data, store intermediate results, and send results back to the Driver.

In simple words:

1. Driver plans the work.
2. Cluster Manager provides resources.
3. Executors perform the actual processing.

## Q2: How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets?

Spark does not execute transformations immediately. It waits until an action is called.

This is called **Lazy Evaluation**.

For example, when we apply multiple operations like filter, select, rename, and cast, Spark first creates an optimized execution plan instead of running every step separately.

Lazy Evaluation improves performance because Spark can:

1. Combine operations
2. Remove unnecessary steps
3. Apply filters early
4. Reduce data movement
5. Execute the final job more efficiently

This is very useful for large datasets because it saves memory, time, and processing cost.

## Q3: Write a Spark command to read a CSV file located at `data/source.csv`, ensuring the first row is treated as a header and inferSchema is enabled.

The code below reads the CSV file using Spark.

- `header=True` treats the first row as column names.
- `inferSchema=True` automatically detects column data types.

In [6]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/source.csv")

print("DataFrame loaded successfully.")
df.show(5)

DataFrame loaded successfully.
+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+
|product_id|product_name|   category|price| old_name|   status|amount|region|priority|base_price|user_id|
+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+
|      P101|      Laptop|Electronics|55000|Product_A|Completed|  1200| North|  Medium|     50000|   U001|
|      P102|       Phone|Electronics|25000|Product_B|  Pending|   800| South|    High|     22000|   U002|
|      P103|       Table|  Furniture| 7000|Product_C|Completed|  1500|  East|     Low|      6500|   NULL|
|      P104|  Headphones|Electronics| 3000|Product_D|Completed|  2000|  West|    High|      2800|   U004|
|      P105|       Chair|  Furniture| 4500|Product_E|Cancelled|   500| North|     Low|      4000|   U005|
+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+



In [7]:
print("Schema of the DataFrame:")
df.printSchema()

Schema of the DataFrame:
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- old_name: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- base_price: integer (nullable = true)
 |-- user_id: string (nullable = true)



## Q4: What is the difference between CSV and Parquet in terms of storage and why does it matter for performance?

CSV is a **row-based** file format. It stores data row by row in plain text format. CSV is easy to read, but it is not very efficient for big data processing.

Parquet is a **columnar** file format. It stores data column by column. This is better for analytical queries because Spark can read only the columns required for the query.

For example, if a dataset has 100 columns and we only need 3 columns, Parquet can read only those 3 columns.

This matters for performance because Parquet:

1. Reduces disk reading
2. Uses less memory
3. Supports compression better
4. Improves analytical query performance
5. Supports optimizations like Predicate Pushdown

## Q5: Given a DataFrame `df`, write a query to select the columns `product_id` and `price` where the category is `Electronics`.

This query selects the required columns and filters rows where category is Electronics.

In [8]:
electronics_df = df.select("product_id", "price") \
    .filter(col("category") == "Electronics")

electronics_df.show()

+----------+-----+
|product_id|price|
+----------+-----+
|      P101|55000|
|      P102|25000|
|      P104| 3000|
+----------+-----+



In [9]:
save_spark_df_as_single_csv(
    spark_df=electronics_df,
    folder_path="outputs/csv_outputs/electronics_products",
    file_name="electronics_products.csv"
)

CSV output saved successfully.
Folder: outputs/csv_outputs/electronics_products
File: electronics_products.csv
Rows saved: 3
Files inside folder: ['electronics_products.csv']


## Q6: Write the code to revise a DataFrame by renaming the column `old_name` to `new_name` and casting the `price` column from String to Double.

In Spark:

- `withColumnRenamed()` is used to rename a column.
- `withColumn()` with `cast()` is used to change the data type of a column.

In [10]:
revised_df = df.withColumnRenamed("old_name", "new_name") \
    .withColumn("price", col("price").cast("double"))

revised_df.show()
revised_df.printSchema()

+----------+------------+-----------+-------+---------+---------+------+------+--------+----------+-------+
|product_id|product_name|   category|  price| new_name|   status|amount|region|priority|base_price|user_id|
+----------+------------+-----------+-------+---------+---------+------+------+--------+----------+-------+
|      P101|      Laptop|Electronics|55000.0|Product_A|Completed|  1200| North|  Medium|     50000|   U001|
|      P102|       Phone|Electronics|25000.0|Product_B|  Pending|   800| South|    High|     22000|   U002|
|      P103|       Table|  Furniture| 7000.0|Product_C|Completed|  1500|  East|     Low|      6500|   NULL|
|      P104|  Headphones|Electronics| 3000.0|Product_D|Completed|  2000|  West|    High|      2800|   U004|
|      P105|       Chair|  Furniture| 4500.0|Product_E|Cancelled|   500| North|     Low|      4000|   U005|
+----------+------------+-----------+-------+---------+---------+------+------+--------+----------+-------+

root
 |-- product_id: strin

In [11]:
save_spark_df_as_single_csv(
    spark_df=revised_df,
    folder_path="outputs/csv_outputs/revised_dataframe",
    file_name="revised_dataframe.csv"
)

CSV output saved successfully.
Folder: outputs/csv_outputs/revised_dataframe
File: revised_dataframe.csv
Rows saved: 5
Files inside folder: ['revised_dataframe.csv']


## Q7: How does Spark use the Lineage Graph or DAG to provide fault tolerance if a worker node fails?

Spark keeps track of every transformation using a **Lineage Graph**, also called a **DAG**.

DAG stands for Directed Acyclic Graph.

If a worker node fails, Spark does not need to restart the complete job from the beginning. Instead, it checks the DAG and recomputes only the lost partitions from the original data and transformation steps.

This makes Spark fault-tolerant because Spark remembers how each DataFrame was created.

## Q8: Write a query to filter a DataFrame `df_orders` for rows where the status is `Completed` AND the amount is greater than `1000`.

This query uses two conditions:

1. `status` should be `Completed`
2. `amount` should be greater than `1000`

Both conditions must be true, so the AND operator `&` is used.

In [12]:
df_orders = df

completed_orders_df = df_orders.filter(
    (col("status") == "Completed") & (col("amount") > 1000)
)

completed_orders_df.show()

+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+
|product_id|product_name|   category|price| old_name|   status|amount|region|priority|base_price|user_id|
+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+
|      P101|      Laptop|Electronics|55000|Product_A|Completed|  1200| North|  Medium|     50000|   U001|
|      P103|       Table|  Furniture| 7000|Product_C|Completed|  1500|  East|     Low|      6500|   NULL|
|      P104|  Headphones|Electronics| 3000|Product_D|Completed|  2000|  West|    High|      2800|   U004|
+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+



In [13]:
save_spark_df_as_single_csv(
    spark_df=completed_orders_df,
    folder_path="outputs/csv_outputs/completed_orders",
    file_name="completed_orders.csv"
)

CSV output saved successfully.
Folder: outputs/csv_outputs/completed_orders
File: completed_orders.csv
Rows saved: 3
Files inside folder: ['completed_orders.csv']


## Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

Predicate Pushdown means Spark pushes filter conditions closer to the data source before loading the data into memory.

In Parquet files, Spark can use metadata to skip unnecessary row groups.

For example, if the filter condition is `region = 'North'`, Spark can skip file blocks that do not contain North region records.

This reduces:

1. Data loaded into memory
2. Disk I/O
3. Processing time
4. Overall resource usage

Predicate Pushdown is one reason Parquet performs better than CSV for analytical workloads.

## Q10: Write a code snippet to add a new column `final_price` which is the `base_price` multiplied by `1.18`.

This adds an 18% tax to the base price.

Formula:

`final_price = base_price * 1.18`

In [14]:
final_price_df = df.withColumn(
    "final_price",
    col("base_price") * 1.18
)

final_price_df.show()

+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+-----------+
|product_id|product_name|   category|price| old_name|   status|amount|region|priority|base_price|user_id|final_price|
+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+-----------+
|      P101|      Laptop|Electronics|55000|Product_A|Completed|  1200| North|  Medium|     50000|   U001|    59000.0|
|      P102|       Phone|Electronics|25000|Product_B|  Pending|   800| South|    High|     22000|   U002|    25960.0|
|      P103|       Table|  Furniture| 7000|Product_C|Completed|  1500|  East|     Low|      6500|   NULL|     7670.0|
|      P104|  Headphones|Electronics| 3000|Product_D|Completed|  2000|  West|    High|      2800|   U004|     3304.0|
|      P105|       Chair|  Furniture| 4500|Product_E|Cancelled|   500| North|     Low|      4000|   U005|     4720.0|
+----------+------------+-----------+-----+---------+---

In [15]:
save_spark_df_as_single_csv(
    spark_df=final_price_df,
    folder_path="outputs/csv_outputs/final_price_data",
    file_name="final_price_data.csv"
)

CSV output saved successfully.
Folder: outputs/csv_outputs/final_price_data
File: final_price_data.csv
Rows saved: 5
Files inside folder: ['final_price_data.csv']


## Q11: What is the difference between Transformations and Actions? Provide two examples of each.

### Transformations

Transformations create a new DataFrame from an existing DataFrame. They are lazy, so they do not execute immediately.

Examples:

1. `filter()`
2. `select()`
3. `withColumn()`
4. `withColumnRenamed()`

### Actions

Actions trigger the actual execution of a Spark job.

Examples:

1. `show()`
2. `count()`
3. `collect()`
4. `write()`

In simple words, transformations define the work and actions execute the work.

In [16]:
# Transformation Example 1
electronics_only = df.filter(col("category") == "Electronics")

# Transformation Example 2
selected_columns = df.select("product_id", "price")

print("Two transformation examples created successfully.")

Two transformation examples created successfully.


In [17]:
# Action Example 1
electronics_only.show()

# Action Example 2
total_rows = df.count()

print("Total rows in DataFrame:", total_rows)

+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+
|product_id|product_name|   category|price| old_name|   status|amount|region|priority|base_price|user_id|
+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+
|      P101|      Laptop|Electronics|55000|Product_A|Completed|  1200| North|  Medium|     50000|   U001|
|      P102|       Phone|Electronics|25000|Product_B|  Pending|   800| South|    High|     22000|   U002|
|      P104|  Headphones|Electronics| 3000|Product_D|Completed|  2000|  West|    High|      2800|   U004|
+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+

Total rows in DataFrame: 5


## Q12: Write the Spark command to load a Parquet file from `path/to/input`, filter out any rows where `user_id` is null, and save the result as a CSV at `path/to/output`.

The standard Spark command is:

```python
parquet_df = spark.read.parquet("path/to/input")

filtered_df = parquet_df.filter(col("user_id").isNotNull())

filtered_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("path/to/output")
```

For this Windows notebook, we will still perform the Spark DataFrame filtering, but we will save output using the safe helper function to avoid Hadoop write errors.

In [18]:
# Creating a Parquet input file safely inside the outputs folder
save_spark_df_as_single_parquet(
    spark_df=df,
    folder_path="outputs/parquet_outputs/parquet_input",
    file_name="source_data.parquet"
)

Parquet output saved successfully.
Folder: outputs/parquet_outputs/parquet_input
File: source_data.parquet
Rows saved: 5
Files inside folder: ['source_data.parquet']


In [19]:
# Reading the Parquet file back as a Spark DataFrame
parquet_file_path = "outputs/parquet_outputs/parquet_input/source_data.parquet"

parquet_df = read_parquet_as_spark_df(parquet_file_path)

print("Parquet data loaded successfully.")
parquet_df.show(5)

Parquet data loaded successfully.
+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+
|product_id|product_name|   category|price| old_name|   status|amount|region|priority|base_price|user_id|
+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+
|      P101|      Laptop|Electronics|55000|Product_A|Completed|  1200| North|  Medium|     50000|   U001|
|      P102|       Phone|Electronics|25000|Product_B|  Pending|   800| South|    High|     22000|   U002|
|      P103|       Table|  Furniture| 7000|Product_C|Completed|  1500|  East|     Low|      6500|   NULL|
|      P104|  Headphones|Electronics| 3000|Product_D|Completed|  2000|  West|    High|      2800|   U004|
|      P105|       Chair|  Furniture| 4500|Product_E|Cancelled|   500| North|     Low|      4000|   U005|
+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+



In [20]:
# Filtering rows where user_id is not null
filtered_user_df = parquet_df.filter(col("user_id").isNotNull())

filtered_user_df.show()

+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+
|product_id|product_name|   category|price| old_name|   status|amount|region|priority|base_price|user_id|
+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+
|      P101|      Laptop|Electronics|55000|Product_A|Completed|  1200| North|  Medium|     50000|   U001|
|      P102|       Phone|Electronics|25000|Product_B|  Pending|   800| South|    High|     22000|   U002|
|      P104|  Headphones|Electronics| 3000|Product_D|Completed|  2000|  West|    High|      2800|   U004|
|      P105|       Chair|  Furniture| 4500|Product_E|Cancelled|   500| North|     Low|      4000|   U005|
+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+



In [21]:
# Saving filtered result as CSV
save_spark_df_as_single_csv(
    spark_df=filtered_user_df,
    folder_path="outputs/final_outputs/filtered_users_csv",
    file_name="filtered_users.csv"
)

CSV output saved successfully.
Folder: outputs/final_outputs/filtered_users_csv
File: filtered_users.csv
Rows saved: 4
Files inside folder: ['filtered_users.csv']


In [22]:
# Saving filtered result as Parquet also
save_spark_df_as_single_parquet(
    spark_df=filtered_user_df,
    folder_path="outputs/final_outputs/filtered_users_parquet",
    file_name="filtered_users.parquet"
)

Parquet output saved successfully.
Folder: outputs/final_outputs/filtered_users_parquet
File: filtered_users.parquet
Rows saved: 4
Files inside folder: ['filtered_users.parquet']


## Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

In **Client Mode**, the Driver runs on the same machine where the Spark application is submitted.

This is useful for development and debugging because logs are visible on the local machine.

In **Cluster Mode**, the Driver runs inside the cluster.

This is better for production because the application does not depend on the local machine after submission.

### Simple Difference

1. Client Mode is commonly used for development and testing.
2. Cluster Mode is commonly used for production jobs.
3. In Client Mode, the local machine must stay active.
4. In Cluster Mode, the cluster manages the Driver.

## Q14: Write a query to filter a dataset for rows where the region is `North` OR the priority is `High`.

This query uses two conditions:

1. `region` should be `North`
2. `priority` should be `High`

Only one condition needs to be true, so the OR operator `|` is used.

In [23]:
region_priority_df = df.filter(
    (col("region") == "North") | (col("priority") == "High")
)

region_priority_df.show()

+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+
|product_id|product_name|   category|price| old_name|   status|amount|region|priority|base_price|user_id|
+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+
|      P101|      Laptop|Electronics|55000|Product_A|Completed|  1200| North|  Medium|     50000|   U001|
|      P102|       Phone|Electronics|25000|Product_B|  Pending|   800| South|    High|     22000|   U002|
|      P104|  Headphones|Electronics| 3000|Product_D|Completed|  2000|  West|    High|      2800|   U004|
|      P105|       Chair|  Furniture| 4500|Product_E|Cancelled|   500| North|     Low|      4000|   U005|
+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+



In [24]:
save_spark_df_as_single_csv(
    spark_df=region_priority_df,
    folder_path="outputs/csv_outputs/region_priority_data",
    file_name="region_priority_data.csv"
)

CSV output saved successfully.
Folder: outputs/csv_outputs/region_priority_data
File: region_priority_data.csv
Rows saved: 4
Files inside folder: ['region_priority_data.csv']


## Q15: When exploring a dataset, why is it safer to use `.show(5)` instead of `.collect()` on a multi-terabyte dataset?

`.show(5)` displays only the first 5 rows of the DataFrame.

It is safer because it does not bring the complete dataset to the Driver machine.

`.collect()` brings all rows from all Executors to the Driver. On a multi-terabyte dataset, this can overload memory and crash the Driver.

That is why `.show(5)` is better for quick data exploration on large datasets.

In [25]:
# Safe preview of the dataset
df.show(5)

+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+
|product_id|product_name|   category|price| old_name|   status|amount|region|priority|base_price|user_id|
+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+
|      P101|      Laptop|Electronics|55000|Product_A|Completed|  1200| North|  Medium|     50000|   U001|
|      P102|       Phone|Electronics|25000|Product_B|  Pending|   800| South|    High|     22000|   U002|
|      P103|       Table|  Furniture| 7000|Product_C|Completed|  1500|  East|     Low|      6500|   NULL|
|      P104|  Headphones|Electronics| 3000|Product_D|Completed|  2000|  West|    High|      2800|   U004|
|      P105|       Chair|  Furniture| 4500|Product_E|Cancelled|   500| North|     Low|      4000|   U005|
+----------+------------+-----------+-----+---------+---------+------+------+--------+----------+-------+



## Final Output Check

This cell checks the complete `outputs` folder and confirms that the assignment output files were generated.

Expected important files:

1. `outputs/csv_outputs/electronics_products/electronics_products.csv`
2. `outputs/csv_outputs/revised_dataframe/revised_dataframe.csv`
3. `outputs/csv_outputs/completed_orders/completed_orders.csv`
4. `outputs/csv_outputs/final_price_data/final_price_data.csv`
5. `outputs/parquet_outputs/parquet_input/source_data.parquet`
6. `outputs/final_outputs/filtered_users_csv/filtered_users.csv`
7. `outputs/final_outputs/filtered_users_parquet/filtered_users.parquet`
8. `outputs/csv_outputs/region_priority_data/region_priority_data.csv`

In [26]:
print("Final outputs folder structure:\n")

for root, dirs, files in os.walk("outputs"):
    level = root.replace("outputs", "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")

    sub_indent = " " * 4 * (level + 1)
    for file in files:
        print(f"{sub_indent}{file}")

Final outputs folder structure:

outputs/
    csv_outputs/
        completed_orders/
            completed_orders.csv
        electronics_products/
            electronics_products.csv
        final_price_data/
            final_price_data.csv
        region_priority_data/
            region_priority_data.csv
        revised_dataframe/
            revised_dataframe.csv
    final_outputs/
        filtered_users_csv/
            filtered_users.csv
        filtered_users_parquet/
            filtered_users.parquet
    parquet_outputs/
        parquet_input/
            source_data.parquet


## Final Insights

This assignment helped in understanding Spark architecture and efficient data processing.

Key learnings:

1. Spark uses Driver, Cluster Manager, and Executors to distribute processing.
2. Lazy Evaluation helps Spark optimize the execution plan before running jobs.
3. DAG or Lineage Graph helps Spark recover lost data if a worker node fails.
4. CSV is row-based, while Parquet is columnar and better for analytics.
5. Predicate Pushdown reduces the amount of data loaded into memory.
6. Transformations are lazy, while actions trigger execution.
7. Filtering and selecting required columns early improves performance.
8. `.show(5)` is safer than `.collect()` for exploring large datasets.
9. All practical outputs were saved directly inside the `outputs` folder.

In [27]:
summary_text = '''Week 6 Spark Assignment - Execution Summary

The notebook executed Spark DataFrame operations for:
1. Reading CSV with header and inferSchema.
2. Selecting Electronics products.
3. Renaming old_name to new_name.
4. Casting price to Double.
5. Filtering Completed orders with amount greater than 1000.
6. Adding final_price with 18% tax.
7. Creating and reading Parquet data safely.
8. Filtering rows where user_id is not null.
9. Filtering rows where region is North OR priority is High.
10. Saving output files inside the outputs folder.

Performance and Architecture Insights:
- Spark Driver plans the execution.
- Cluster Manager provides resources.
- Executors run tasks.
- Lazy Evaluation optimizes chained transformations.
- DAG provides fault tolerance.
- Parquet improves analytics performance because of columnar storage and Predicate Pushdown.
- show(5) is safer than collect() for large datasets.
'''

summary_path = Path("outputs/execution_results_summary.txt")
summary_path.write_text(summary_text, encoding="utf-8")

print("Execution summary saved at:", summary_path)

Execution summary saved at: outputs\execution_results_summary.txt


## Stop Spark Session

This is the final cell.  
It stops the Spark session cleanly after all tasks are complete.

In [28]:
spark.stop()
print("Spark Session stopped successfully.")

Spark Session stopped successfully.
